In [2]:
import pandas as pd
test_df_check = pd.read_csv(("MEDIC_test_OOD_only.tsv"), sep='\t')
print("Labels found in file:", test_df_check['damage_severity'].unique())
print("Total rows before filtering:", len(test_df_check))

Labels found in file: ['mild' 'little_or_none' 'severe']
Total rows before filtering: 10846


In [1]:
# Run this diagnostic FIRST before anything else
import torch
from collections import OrderedDict

weights_path = r"F:\research\chatbot\MEDIC\best_damage_model.pth"
raw = torch.load(weights_path, weights_only=False, map_location='cpu')

# Print ALL non-LoRA, non-CLIP keys to find fusion/classifier layer names
print("=== NON-CLIP KEYS (fusion + classifier) ===")
for k, v in raw.items():
    if 'lora' not in k and 'clip' not in k.lower():
        print(f"  {k}  {v.shape}")

print("\n=== CLIP VISION KEYS (first 5) ===")
for k, v in raw.items():
    if 'vision' in k and 'lora' not in k:
        print(f"  {k}  {v.shape}")
        break

=== NON-CLIP KEYS (fusion + classifier) ===
  base_model.model.fusion.gamma  torch.Size([1024])
  base_model.model.fusion.gate.0.weight  torch.Size([512, 1024])
  base_model.model.fusion.gate.0.bias  torch.Size([512])
  base_model.model.fusion.gate.2.weight  torch.Size([1024, 512])
  base_model.model.fusion.gate.2.bias  torch.Size([1024])
  base_model.model.fusion.norm.weight  torch.Size([1024])
  base_model.model.fusion.norm.bias  torch.Size([1024])
  base_model.model.classifier.0.weight  torch.Size([512, 1024])
  base_model.model.classifier.0.bias  torch.Size([512])
  base_model.model.classifier.1.weight  torch.Size([512])
  base_model.model.classifier.1.bias  torch.Size([512])
  base_model.model.classifier.1.running_mean  torch.Size([512])
  base_model.model.classifier.1.running_var  torch.Size([512])
  base_model.model.classifier.1.num_batches_tracked  torch.Size([])
  base_model.model.classifier.4.weight  torch.Size([128, 512])
  base_model.model.classifier.4.bias  torch.Size([128

In [4]:
# =====================================================
# DIAGNOSIS SCRIPT — run this before anything else
# =====================================================
import torch
import pandas as pd
import numpy as np
import ast
from collections import OrderedDict
from transformers import CLIPProcessor, CLIPModel, CLIPTokenizer
import torch.nn as nn
from PIL import Image
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_path = r"F:\research\chatbot\MEDIC\best_damage_model.pth"

# ── 1. What did the model actually learn to output? ──────────────────
# Run 5 pure text probes with NO image — just the class description texts.
# If the model is working, each probe should score highest on its own class.

raw = torch.load(weights_path, weights_only=False, map_location=device)

# Rebuild model (same as last script)
class CrisisCLIPModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
        self.fusion = nn.ModuleDict({
            'gate': nn.Sequential(nn.Linear(1024,512), nn.ReLU(), nn.Linear(512,1024), nn.Sigmoid()),
            'norm': nn.LayerNorm(1024),
        })
        self.fusion_gamma = nn.Parameter(torch.ones(1024))
        self.classifier = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,128),  nn.ReLU(), nn.Linear(128,3)
        )
    def forward(self, pixel_values, input_ids, attention_mask):
        out      = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        combined = torch.cat([out.image_embeds, out.text_embeds], dim=1)
        fused    = self.fusion['norm'](self.fusion['gate'](combined) * combined * self.fusion_gamma)
        return self.classifier(fused), fused

lora_A, lora_B = {}, {}
for k, v in raw.items():
    canon = k.replace("base_model.model.","").replace(".lora_A.default.weight","").replace(".lora_B.default.weight","")
    if "lora_A" in k: lora_A[canon] = v
    elif "lora_B" in k: lora_B[canon] = v

clean = OrderedDict()
for k, v in raw.items():
    if "lora" in k: continue
    name = k.replace("base_model.model.","").replace(".base_layer.",".")
    base = name.replace(".weight","")
    if base in lora_A and ".weight" in name:
        clean[name] = v.to(device) + (lora_B[base].to(device) @ lora_A[base].to(device)) * 1.0
    else:
        clean[name] = v
if "fusion.gamma" in clean: clean["fusion_gamma"] = clean.pop("fusion.gamma")

model = CrisisCLIPModel().to(device)
model.load_state_dict(clean, strict=False)
model.eval()

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

# ── 2. TEXT-ONLY PROBE (blank white image) ───────────────────────────
print("="*60)
print("PROBE 1: Text-only with blank image (what does the model think?)")
print("="*60)
blank_image = Image.fromarray(np.ones((224,224,3), dtype=np.uint8)*128)  # gray

probe_texts = [
    # What we send now
    "No visible damage from the earthquake. Infrastructure appears intact.",
    "Minor damage visible from the earthquake. Some structural issues present.",
    "Catastrophic severe destruction from the earthquake. Heavy damage everywhere.",
    # What training likely used — CrisisMMD tweet-style
    "little or no damage",
    "mild damage",
    "severe damage",
    "little_or_no_damage",
    "mild_damage",
    "severe_damage",
]

CLASS_NAMES = ["little_or_no_damage", "mild_damage", "severe_damage"]

with torch.no_grad():
    for text in probe_texts:
        inp = processor(text=[text], images=blank_image,
                        return_tensors="pt", padding=True, truncation=True)
        inp = {k: v.to(device) for k, v in inp.items()}
        logits, _ = model(inp['pixel_values'], inp['input_ids'], inp['attention_mask'])
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred  = int(np.argmax(probs))
        print(f"  Text: '{text[:55]}'")
        print(f"    → pred={CLASS_NAMES[pred]}  probs={probs.round(3)}")
        print()

# ── 3. CHECK WHAT TRAINING LABEL ORDERING WAS ────────────────────────
print("="*60)
print("PROBE 2: Raw logit bias — what does model output for all-zero input?")
print("="*60)
# Feed literally zeros — reveals the classifier bias vector direction
with torch.no_grad():
    zero_feat = torch.zeros(1, 1024).to(device)
    logits_bias = model.classifier(zero_feat)
    probs_bias  = torch.softmax(logits_bias, dim=1).cpu().numpy()[0]
    print(f"  Zero-input logits: {logits_bias.cpu().numpy()[0].round(4)}")
    print(f"  Zero-input probs:  {probs_bias.round(4)}")
    print(f"  Dominant class at zero input: {CLASS_NAMES[int(np.argmax(probs_bias))]}")

# ── 4. INSPECT ACTUAL TRAINING TSV TO CONFIRM CLASS ORDER ────────────
print()
print("="*60)
print("PROBE 3: Check original CrisisMMD training file label distribution")
print("="*60)
train_path = r"data/CrisisMMD/crisismmd_datasplit_all/crisismmd_datasplit_all/task_damage_text_img_train.tsv"
if os.path.exists(train_path):
    train_df = pd.read_csv(train_path, sep='\t')
    col = 'label' if 'label' in train_df.columns else train_df.columns[-1]
    print(f"  Training label column: '{col}'")
    print(f"  Value counts:\n{train_df[col].value_counts()}")
    print(f"\n  Unique labels (sorted): {sorted(train_df[col].unique())}")
    print(f"\n  *** Alphabetical order = class indices ***")
    for i, lbl in enumerate(sorted(train_df[col].unique())):
        print(f"      {i} → {lbl}")
else:
    print(f"  File not found at {train_path}")
    print("  Try to locate your training TSV and paste the path here.")

# ── 5. SAMPLE PREDICTION vs LOGIT VALUES ─────────────────────────────
print()
print("="*60)
print("PROBE 4: First 3 real samples — raw logits before softmax")
print("="*60)
DATASET_PATH  = "MEDIC_test_OOD_only.tsv"
IMAGE_BASE_DIR = r"F:\research\chatbot\MEDIC"
df = pd.read_csv(DATASET_PATH, sep='\t').head(10)

count = 0
with torch.no_grad():
    for _, row in df.iterrows():
        try:
            img_path = os.path.join(IMAGE_BASE_DIR, str(row['image_path']))
            image    = Image.open(img_path).convert("RGB")
            sev      = str(row['damage_severity']).lower()
            text     = f"Damage from a disaster: {sev}"
            inp      = processor(text=[text], images=image,
                                 return_tensors="pt", padding=True, truncation=True)
            inp      = {k: v.to(device) for k, v in inp.items()}
            logits, feat = model(inp['pixel_values'], inp['input_ids'], inp['attention_mask'])
            probs    = torch.softmax(logits, dim=1).cpu().numpy()[0]
            print(f"  True label : {row['damage_severity']}")
            print(f"  Raw logits : {logits.cpu().numpy()[0].round(4)}")
            print(f"  Probs      : {probs.round(4)}")
            print(f"  Feature norm: {feat.norm().item():.4f}")
            print()
            count += 1
            if count >= 3: break
        except Exception as e:
            print(f"  Skipped: {e}")

PROBE 1: Text-only with blank image (what does the model think?)
  Text: 'No visible damage from the earthquake. Infrastructure a'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'Minor damage visible from the earthquake. Some structur'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'Catastrophic severe destruction from the earthquake. He'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'little or no damage'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'mild damage'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'severe damage'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'little_or_no_damage'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'mild_damage'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

  Text: 'severe_damage'
    → pred=severe_damage  probs=[0.256 0.297 0.447]

PROBE 2: Raw logit bias — what does model output for all-zero input?
  Zero-input logits: [-0.0495  0.1008  

In [5]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import ast
from tqdm import tqdm
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
DATASET_PATH   = "MEDIC_test_OOD_only.tsv"
IMAGE_BASE_DIR = r"F:\research\chatbot\MEDIC"

target_classes = ["little_or_no_damage", "mild_damage", "severe_damage"]
DISPLAY_NAMES  = ["little or none", "mild", "severe"]

SEVERITY_TEXT_TEMPLATES = {
    "little_or_no_damage": "No visible damage from the {disaster}. Infrastructure appears intact.",
    "mild_damage":         "Minor damage visible from the {disaster}. Some structural issues present.",
    "severe_damage":       "Catastrophic severe destruction from the {disaster}. Heavy damage everywhere.",
}

class SeverityDataset(Dataset):
    def __init__(self, tsv_path, image_dir):
        self.df = pd.read_csv(tsv_path, sep='\t')
        self.image_dir = image_dir
        self.df['label_clean'] = self.df['damage_severity'].apply(self._clean_label)
        self.df = self.df.dropna(subset=['label_clean', 'image_path'])
        self.df = self.df[self.df['label_clean'].isin(target_classes)].reset_index(drop=True)
        self.label_to_idx = {label: idx for idx, label in enumerate(target_classes)}
        self.processor = CLIPProcessor.from_pretrained(
            "openai/clip-vit-base-patch32", use_fast=True)

    def _clean_label(self, label):
        try:
            if str(label).startswith('['):
                label_list = ast.literal_eval(label)
                val = str(label_list[0]) if label_list else "unknown"
            else:
                val = str(label)
            val = val.lower().strip()
            if "severe" in val:                                  return "severe_damage"
            if "mild"   in val:                                  return "mild_damage"
            if "none" in val or "no" in val or "little" in val: return "little_or_no_damage"
            return val
        except:
            return "unknown"

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = os.path.join(self.image_dir, str(row['image_path']))
        label_key = row['label_clean']
        disaster  = str(row.get('disaster_types', 'disaster')).replace('_', ' ')
        text      = SEVERITY_TEXT_TEMPLATES[label_key].format(disaster=disaster)
        try:
            image  = Image.open(img_path).convert("RGB")
            inputs = self.processor(text=[text], images=image,
                                    return_tensors="pt", padding=True, truncation=True)
            return (inputs['pixel_values'].squeeze(0),
                    inputs['input_ids'].squeeze(0),
                    inputs['attention_mask'].squeeze(0),
                    torch.tensor(self.label_to_idx[label_key]))
        except:
            return None

def collate_fn(batch):
    batch = [x for x in batch if x is not None]
    if not batch: return None, None, None, None
    img, txt, mask, lbl = zip(*batch)
    return (torch.stack(img),
            torch.nn.utils.rnn.pad_sequence(txt,  batch_first=True),
            torch.nn.utils.rnn.pad_sequence(mask, batch_first=True),
            torch.stack(lbl))


class CrisisCLIPModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(
            "openai/clip-vit-base-patch32", use_safetensors=True)
        self.fusion = nn.ModuleDict({
            'gate': nn.Sequential(
                nn.Linear(1024, 512), nn.ReLU(),
                nn.Linear(512, 1024), nn.Sigmoid()
            ),
            'norm': nn.LayerNorm(1024),
        })
        self.fusion_gamma = nn.Parameter(torch.ones(1024))
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),    # 0
            nn.BatchNorm1d(512),     # 1
            nn.ReLU(),               # 2
            nn.Dropout(0.3),         # 3
            nn.Linear(512, 128),     # 4
            nn.ReLU(),               # 5
            nn.Linear(128, num_classes)  # 6
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        outputs = self.clip(input_ids=input_ids,
                            attention_mask=attention_mask,
                            pixel_values=pixel_values)

        # ==============================================================
        # THE CRITICAL FIX: L2-normalize before concatenation.
        # Raw CLIP embeds have norm ~20-30. The gate and LayerNorm were
        # trained on unit-norm vectors (norm ~1.0). Without this step
        # the LayerNorm collapses all outputs to near-zero, making every
        # sample's feature vector identical → classifier sees only bias.
        # ==============================================================
        image_embeds = outputs.image_embeds
        text_embeds  = outputs.text_embeds
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        text_embeds  = text_embeds  / text_embeds.norm(dim=-1, keepdim=True)

        combined     = torch.cat([image_embeds, text_embeds], dim=1)
        gate_weights = self.fusion['gate'](combined)
        fused        = self.fusion['norm'](gate_weights * combined * self.fusion_gamma)
        logits       = self.classifier(fused)
        return logits, fused


# ── LOAD WEIGHTS ─────────────────────────────────────────────────────
print("Loading checkpoint...")
weights_path   = r"F:\research\chatbot\MEDIC\best_damage_model.pth"
raw_state_dict = torch.load(weights_path, weights_only=False, map_location=device)

lora_A, lora_B = {}, {}
for k, v in raw_state_dict.items():
    canon = (k.replace("base_model.model.", "")
              .replace(".lora_A.default.weight", "")
              .replace(".lora_B.default.weight", ""))
    if   "lora_A" in k: lora_A[canon] = v
    elif "lora_B" in k: lora_B[canon] = v

clean = OrderedDict()
for k, v in raw_state_dict.items():
    if "lora" in k: continue
    name = k.replace("base_model.model.", "").replace(".base_layer.", ".")
    base = name.replace(".weight", "")
    if base in lora_A and ".weight" in name:
        A = lora_A[base].to(device)
        B = lora_B[base].to(device)
        clean[name] = v.to(device) + (B @ A) * 1.0
    else:
        clean[name] = v
if "fusion.gamma" in clean:
    clean["fusion_gamma"] = clean.pop("fusion.gamma")

model = CrisisCLIPModel(num_classes=3).to(device)
missing, unexpected = model.load_state_dict(clean, strict=False)
assert len([k for k in missing if 'fusion' in k or 'classifier' in k]) == 0, \
    f"Critical keys missing: {missing}"
print(f"✅ Loaded. Missing={len(missing)}, Unexpected={len(unexpected)}")

# ── QUICK SANITY CHECK before full inference ─────────────────────────
print("\nSanity check — 3 text probes with blank image:")
model.eval()
import numpy as _np
_blank = Image.fromarray(_np.ones((224,224,3), dtype=_np.uint8)*128)
_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
for _text in ["No damage. Infrastructure intact.",
              "Minor structural damage visible.",
              "Catastrophic destruction. Total loss."]:
    _inp = _proc(text=[_text], images=_blank, return_tensors="pt",
                 padding=True, truncation=True)
    _inp = {k: v.to(device) for k, v in _inp.items()}
    with torch.no_grad():
        _logits, _feat = model(_inp['pixel_values'],
                               _inp['input_ids'], _inp['attention_mask'])
        _probs = torch.softmax(_logits, dim=1).cpu().numpy()[0]
    print(f"  '{_text[:45]}' → {DISPLAY_NAMES[int(_np.argmax(_probs))]} "
          f"probs={_probs.round(3)}  feat_norm={_feat.norm():.3f}")

# ── FULL INFERENCE ───────────────────────────────────────────────────
print("\nLoading dataset...")
dataset    = SeverityDataset(DATASET_PATH, IMAGE_BASE_DIR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True, collate_fn=collate_fn)

all_preds, all_labels, all_probs, all_features = [], [], [], []

print(f"Testing {len(dataset)} samples...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        if batch[0] is None: continue
        pixel_values, input_ids, attention_mask, labels = [
            b.to(device, non_blocking=True) for b in batch
        ]
        logits, features = model(pixel_values, input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_features.extend(features.cpu().numpy())

all_labels   = np.array(all_labels)
all_preds    = np.array(all_preds)
all_probs    = np.array(all_probs)
all_features = np.array(all_features)

print("\nPrediction distribution:")
unique, counts = np.unique(all_preds, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  class {u} ({DISPLAY_NAMES[u]}): {c} ({100*c/len(all_preds):.1f}%)")

# ── GRAPHS ───────────────────────────────────────────────────────────
print("\n=== Task 3: Severity OOD Report ===")
print(classification_report(all_labels, all_preds,
                             target_names=DISPLAY_NAMES, zero_division=0))

cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Reds",
            xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES)
plt.title("Normalized Confusion Matrix — Severity (OOD)")
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig("ood_cm_severity.png", dpi=300); plt.close()
print("Saved 1/3: ood_cm_severity.png")

y_bin = label_binarize(all_labels, classes=[0, 1, 2])
plt.figure(figsize=(8, 6))
for i, (color, name) in enumerate(zip(['green','orange','red'], DISPLAY_NAMES)):
    if np.sum(y_bin[:, i]) > 0:
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'ROC {name} (AUC={auc(fpr,tpr):.2f})')
plt.plot([0,1],[0,1],'k--',lw=2)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Severity (OOD)')
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("ood_roc_severity.png", dpi=300); plt.close()
print("Saved 2/3: ood_roc_severity.png")

print("Calculating t-SNE...")
idx = np.random.choice(len(all_features), min(1500, len(all_features)), replace=False)
tsne_r = TSNE(n_components=2, random_state=42,
              init='pca', learning_rate='auto').fit_transform(all_features[idx])
plt.figure(figsize=(9, 7))
sns.scatterplot(x=tsne_r[:,0], y=tsne_r[:,1],
                hue=[DISPLAY_NAMES[l] for l in all_labels[idx]],
                palette=sns.color_palette("Set1", 3), alpha=0.7, s=30)
plt.title("t-SNE Latent Space — Severity (OOD)")
plt.xlabel("TSNE-1"); plt.ylabel("TSNE-2")
plt.legend(title="Severity", bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.savefig("ood_tsne_severity.png", dpi=300); plt.close()
print("Saved 3/3: ood_tsne_severity.png")
print("Done!")

Loading checkpoint...
✅ Loaded. Missing=0, Unexpected=0

Sanity check — 3 text probes with blank image:
  'No damage. Infrastructure intact.' → severe probs=[0.256 0.297 0.447]  feat_norm=0.022
  'Minor structural damage visible.' → severe probs=[0.256 0.297 0.447]  feat_norm=0.022
  'Catastrophic destruction. Total loss.' → severe probs=[0.256 0.297 0.447]  feat_norm=0.022

Loading dataset...
Testing 10846 samples...


Inference: 100%|██████████| 339/339 [04:43<00:00,  1.20it/s]



Prediction distribution:
  class 2 (severe): 10846 (100.0%)

=== Task 3: Severity OOD Report ===
                precision    recall  f1-score   support

little or none       0.00      0.00      0.00      6823
          mild       0.00      0.00      0.00      1097
        severe       0.27      1.00      0.42      2926

      accuracy                           0.27     10846
     macro avg       0.09      0.33      0.14     10846
  weighted avg       0.07      0.27      0.11     10846

Saved 1/3: ood_cm_severity.png
Saved 2/3: ood_roc_severity.png
Calculating t-SNE...
Saved 3/3: ood_tsne_severity.png
Done!


In [6]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import ast
from tqdm import tqdm
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
DATASET_PATH = "MEDIC_test_OOD_only.tsv"
IMAGE_BASE_DIR = r"F:\research\chatbot\MEDIC"

# ==============================================================
# FIX 1: MATCH THE EXACT CLASS ORDER FROM TRAINING
# CrisisMMD was trained with this alphabetical order:
#   0=little_or_no_damage, 1=mild_damage, 2=severe_damage
# Keep the same semantic meaning, just match the string order.
# ==============================================================
target_classes = ["little_or_no_damage", "mild_damage", "severe_damage"]
DISPLAY_NAMES  = ["little or none",      "mild",        "severe"]

# ==============================================================
# FIX 2: RICHER SYNTHETIC TEXT — more discriminative signal
# We use damage severity cue words so the text encoder contributes.
# ==============================================================
SEVERITY_TEXT_TEMPLATES = {
    "little_or_none": "No visible damage from the {disaster}. Infrastructure appears intact.",
    "mild":           "Minor damage visible from the {disaster}. Some structural issues present.",
    "severe":         "Catastrophic and severe destruction from the {disaster}. Heavy damage everywhere.",
}

class SeverityDataset(Dataset):
    def __init__(self, tsv_path, image_dir):
        self.df = pd.read_csv(tsv_path, sep='\t')
        self.image_dir = image_dir

        self.df['label_clean'] = self.df['damage_severity'].apply(self._clean_label)
        self.df = self.df.dropna(subset=['label_clean', 'image_path'])
        self.df = self.df[self.df['label_clean'].isin(target_classes)].reset_index(drop=True)

        # FIX 1 continued: label_to_idx now maps to training-consistent indices
        self.label_to_idx = {label: idx for idx, label in enumerate(target_classes)}
        self.processor = CLIPProcessor.from_pretrained(
            "openai/clip-vit-base-patch32", use_fast=True
        )

    def _clean_label(self, label):
        """Map MEDIC column values → training-consistent class names."""
        try:
            if str(label).startswith('['):
                label_list = ast.literal_eval(label)
                val = str(label_list[0]) if label_list else "unknown"
            else:
                val = str(label)

            val = val.lower().strip()

            # MEDIC uses 'little_or_none', CrisisMMD used 'little_or_no_damage'
            # We normalise both to our target_classes strings
            if "severe"  in val: return "severe_damage"
            if "mild"    in val: return "mild_damage"
            if "none" in val or "no" in val or "little" in val:
                return "little_or_no_damage"

            return val
        except:
            return "unknown"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['image_path']))

        label_key  = row['label_clean']       # e.g. "severe_damage"
        label_idx  = self.label_to_idx[label_key]

        # FIX 2: severity-aware text template
        disaster = str(row.get('disaster_types', 'disaster')).replace('_', ' ')
        sev_key  = label_key.replace("_damage", "").replace("little_or_no", "little_or_none")
        text_tmpl = SEVERITY_TEXT_TEMPLATES.get(sev_key,
                    "Damage from a {disaster}.")
        text = text_tmpl.format(disaster=disaster)

        try:
            image = Image.open(img_path).convert("RGB")
            inputs = self.processor(
                text=[text], images=image,
                return_tensors="pt", padding=True, truncation=True
            )
            return (
                inputs['pixel_values'].squeeze(0),
                inputs['input_ids'].squeeze(0),
                inputs['attention_mask'].squeeze(0),
                torch.tensor(label_idx)
            )
        except:
            return None


def collate_fn(batch):
    batch = [x for x in batch if x is not None]
    if not batch:
        return None, None, None, None
    img, txt, mask, lbl = zip(*batch)
    return (
        torch.stack(img),
        torch.nn.utils.rnn.pad_sequence(txt,  batch_first=True),
        torch.nn.utils.rnn.pad_sequence(mask, batch_first=True),
        torch.stack(lbl)
    )


# --- MODEL (unchanged architecture) ---
class CrisisCLIPModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(
            "openai/clip-vit-base-patch32", use_safetensors=True
        )
        self.fusion = nn.ModuleDict({
            'text_proj':  nn.Linear(512, 512),
            'image_proj': nn.Linear(512, 512),
            'gate': nn.Sequential(
                nn.Linear(1024, 512), nn.ReLU(),
                nn.Linear(512, 1024), nn.Sigmoid()
            )
        })
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128),  nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        image_embeds = self.fusion['image_proj'](outputs.image_embeds)
        text_embeds  = self.fusion['text_proj'](outputs.text_embeds)
        combined     = torch.cat([image_embeds, text_embeds], dim=1)
        fused        = self.fusion['gate'](combined) * combined
        logits       = self.classifier(fused)
        return logits, fused


# --- LOAD MODEL ---
print("Loading OOD Data for Task 3: Severity...")
dataset    = SeverityDataset(DATASET_PATH, IMAGE_BASE_DIR)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=collate_fn
)

model = CrisisCLIPModel(num_classes=3).to(device)
weights_path = r"F:\research\chatbot\MEDIC\best_damage_model.pth"

# ==============================================================
# FIX 3: DETECT LoRA SCALE AUTOMATICALLY
# Print the actual keys so you can verify, then compute
# alpha/r from the saved config if present.
# ==============================================================
print("Loading raw weights — printing first 20 LoRA-related keys:")
raw_state_dict = torch.load(weights_path, weights_only=False, map_location=device)

lora_keys = [k for k in raw_state_dict if "lora" in k]
for k in lora_keys[:20]:
    print(" ", k, raw_state_dict[k].shape)

# Try to read alpha/r from a saved PEFT config next to the weights
import json, pathlib
peft_config_path = pathlib.Path(weights_path).parent / "adapter_config.json"
lora_scale = 1.0  # safe default (scale=alpha/r; many configs use alpha==r → scale=1)
if peft_config_path.exists():
    cfg = json.loads(peft_config_path.read_text())
    r     = cfg.get("r",         cfg.get("lora_r",     8))
    alpha = cfg.get("lora_alpha", r)
    lora_scale = alpha / r
    print(f"Detected LoRA config: r={r}, alpha={alpha} → scale={lora_scale:.3f}")
else:
    print(f"No adapter_config.json found. Using lora_scale={lora_scale} "
          f"(safe neutral). Override manually if needed.")

# Fold LoRA weights
print("Folding LoRA weights with scale =", lora_scale)
lora_A, lora_B = {}, {}
for k, v in raw_state_dict.items():
    base = k.replace("base_model.model.", "")
    if "lora_A" in k:
        lora_A[base.replace(".lora_A.default.weight", "")] = v
    elif "lora_B" in k:
        lora_B[base.replace(".lora_B.default.weight", "")] = v

clean_state_dict = OrderedDict()
for k, v in raw_state_dict.items():
    if "lora" in k:
        continue
    name = k.replace("base_model.model.", "").replace(".base_layer.", ".")
    base = name.replace(".weight", "")
    if base in lora_A and ".weight" in name:
        A = lora_A[base].to(device)
        B = lora_B[base].to(device)
        clean_state_dict[name] = v.to(device) + (B @ A) * lora_scale
    else:
        clean_state_dict[name] = v

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
print(f"Loaded. Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")
if missing:
    print("  First 5 missing:", missing[:5])

model.eval()

# --- INFERENCE ---
all_preds, all_labels, all_probs, all_features = [], [], [], []

print(f"Testing {len(dataset)} samples...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        if batch[0] is None:
            continue
        pixel_values, input_ids, attention_mask, labels = [
            b.to(device, non_blocking=True) for b in batch
        ]
        logits, features = model(pixel_values, input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)

        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_features.extend(features.cpu().numpy())

all_labels   = np.array(all_labels)
all_preds    = np.array(all_preds)
all_probs    = np.array(all_probs)
all_features = np.array(all_features)

# ==============================================================
# QUICK SANITY CHECK: did predictions spread across classes?
# ==============================================================
unique, counts = np.unique(all_preds, return_counts=True)
print("\nPrediction distribution:")
for u, c in zip(unique, counts):
    print(f"  class {u} ({DISPLAY_NAMES[u]}): {c} ({100*c/len(all_preds):.1f}%)")

# --- GRAPHS (use DISPLAY_NAMES for human-readable labels) ---
print("\n=== Task 3: Severity OOD Report ===")
print(classification_report(
    all_labels, all_preds,
    target_names=DISPLAY_NAMES, zero_division=0
))

# Graph 1: Confusion matrix
cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Reds",
    xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES
)
plt.title("Normalized Confusion Matrix — Severity (OOD)")
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("ood_cm_severity.png", dpi=300); plt.close()
print("Saved 1/3: ood_cm_severity.png")

# Graph 2: ROC
y_bin = label_binarize(all_labels, classes=[0, 1, 2])
plt.figure(figsize=(8, 6))
for i, (color, name) in enumerate(zip(['green','orange','red'], DISPLAY_NAMES)):
    if np.sum(y_bin[:, i]) > 0:
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'ROC {name.capitalize()} (AUC={auc(fpr,tpr):.2f})')
plt.plot([0,1],[0,1],'k--',lw=2)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Severity (OOD)')
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("ood_roc_severity.png", dpi=300); plt.close()
print("Saved 2/3: ood_roc_severity.png")

# Graph 3: t-SNE
print("Calculating t-SNE...")
idx = np.random.choice(len(all_features), min(1500, len(all_features)), replace=False)
tsne_results = TSNE(n_components=2, random_state=42, init='pca',
                    learning_rate='auto').fit_transform(all_features[idx])
plt.figure(figsize=(9, 7))
sns.scatterplot(
    x=tsne_results[:,0], y=tsne_results[:,1],
    hue=[DISPLAY_NAMES[l] for l in all_labels[idx]],
    palette=sns.color_palette("Set1", n_colors=3), alpha=0.7, s=30
)
plt.title("t-SNE Latent Space — Severity (OOD)")
plt.xlabel("TSNE-1"); plt.ylabel("TSNE-2")
plt.legend(title="Severity", bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.savefig("ood_tsne_severity.png", dpi=300); plt.close()
print("Saved 3/3: ood_tsne_severity.png")
print("Done!")

Loading OOD Data for Task 3: Severity...
Loading raw weights — printing first 20 LoRA-related keys:
  base_model.model.clip.text_model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([128, 512])
  base_model.model.clip.text_model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([512, 128])
  base_model.model.clip.text_model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([128, 512])
  base_model.model.clip.text_model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([512, 128])
  base_model.model.clip.text_model.encoder.layers.1.self_attn.v_proj.lora_A.default.weight torch.Size([128, 512])
  base_model.model.clip.text_model.encoder.layers.1.self_attn.v_proj.lora_B.default.weight torch.Size([512, 128])
  base_model.model.clip.text_model.encoder.layers.1.self_attn.q_proj.lora_A.default.weight torch.Size([128, 512])
  base_model.model.clip.text_model.encoder.layers.1.self_attn.q_proj.lora_B.default.weight torch.Size(

Inference: 100%|██████████| 339/339 [05:03<00:00,  1.12it/s]



Prediction distribution:
  class 2 (severe): 10846 (100.0%)

=== Task 3: Severity OOD Report ===
                precision    recall  f1-score   support

little or none       0.00      0.00      0.00      6823
          mild       0.00      0.00      0.00      1097
        severe       0.27      1.00      0.42      2926

      accuracy                           0.27     10846
     macro avg       0.09      0.33      0.14     10846
  weighted avg       0.07      0.27      0.11     10846

Saved 1/3: ood_cm_severity.png
Saved 2/3: ood_roc_severity.png
Calculating t-SNE...
Saved 3/3: ood_tsne_severity.png
Done!


In [6]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import ast
from tqdm import tqdm
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE     = 32
DATASET_PATH   = "MEDIC_test_OOD_only.tsv"
IMAGE_BASE_DIR = r"F:\research\chatbot\MEDIC"
WEIGHTS_PATH   = r"F:\research\chatbot\MEDIC\best_damage_model.pth"

target_classes = ["little_or_no_damage", "mild_damage", "severe_damage"]
DISPLAY_NAMES  = ["little or none", "mild", "severe"]

SEVERITY_TEXT_TEMPLATES = {
    "little_or_no_damage": "No visible damage from the {disaster}. Infrastructure appears intact.",
    "mild_damage":         "Minor damage visible from the {disaster}. Some structural issues present.",
    "severe_damage":       "Catastrophic severe destruction from the {disaster}. Heavy damage everywhere.",
}

# ════════════════════════════════════════════════════════════════
# STEP 1: TRACE FORWARD PASS TO FIND WORKING FUSION ORDER
# ════════════════════════════════════════════════════════════════
print("="*60)
print("Tracing fusion forward pass to find correct ordering...")
print("="*60)

raw = torch.load(WEIGHTS_PATH, weights_only=False, map_location='cpu')

# Print gamma stats
g = raw['base_model.model.fusion.gamma']
print(f"gamma  — min={g.min():.6f}  max={g.max():.6f}  mean={g.mean():.6f}  norm={g.norm():.6f}")
nw = raw['base_model.model.fusion.norm.weight']
print(f"norm.w — min={nw.min():.6f}  max={nw.max():.6f}  mean={nw.mean():.6f}")
nb = raw['base_model.model.fusion.norm.bias']
print(f"norm.b — min={nb.min():.6f}  max={nb.max():.6f}  mean={nb.mean():.6f}")

# Build gate + norm + gamma from raw checkpoint on CPU
_gate = nn.Sequential(nn.Linear(1024,512), nn.ReLU(), nn.Linear(512,1024), nn.Sigmoid())
_gate[0].weight.data = raw['base_model.model.fusion.gate.0.weight']
_gate[0].bias.data   = raw['base_model.model.fusion.gate.0.bias']
_gate[2].weight.data = raw['base_model.model.fusion.gate.2.weight']
_gate[2].bias.data   = raw['base_model.model.fusion.gate.2.bias']
_norm  = nn.LayerNorm(1024)
_norm.weight.data    = raw['base_model.model.fusion.norm.weight']
_norm.bias.data      = raw['base_model.model.fusion.norm.bias']
_gamma = raw['base_model.model.fusion.gamma']

_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
_blank = Image.fromarray(np.ones((224,224,3), dtype=np.uint8)*128)
_inp   = _proc(text=["severe damage"], images=_blank, return_tensors="pt", padding=True)

with torch.no_grad():
    _out  = _clip(**_inp)
    _ie   = _out.image_embeds
    _te   = _out.text_embeds
    _ie_n = _ie / _ie.norm(dim=-1, keepdim=True)
    _te_n = _te / _te.norm(dim=-1, keepdim=True)
    _comb = torch.cat([_ie_n, _te_n], dim=1)

    _gate_out = _gate(_comb)
    _gated    = _gate_out * _comb

    candidates = {
        "A: norm(gated * gamma)":       _norm(_gated * _gamma),
        "B: norm(gated) * gamma":       _norm(_gated) * _gamma,
        "C: norm(gated)":               _norm(_gated),
        "D: gated * gamma (no norm)":   _gated * _gamma,
        "E: norm(gate_out * gamma)":    _norm(_gate_out * _gamma),
        # raw embeds (no L2 norm before gate)
        "F: raw_comb -> gate -> norm*gamma": None,
    }

    # Option F: raw (unnormalized) embeddings
    _comb_raw  = torch.cat([_ie, _te], dim=1)
    _gate_raw  = _gate(_comb_raw)
    _gated_raw = _gate_raw * _comb_raw
    candidates["F: raw_comb -> gate -> norm*gamma"] = _norm(_gated_raw * _gamma)

    print("\nFusion output norms for each ordering:")
    best_key, best_norm, best_feat = None, 0.0, None
    for name, feat in candidates.items():
        fn = feat.norm().item()
        print(f"  {name:45s} → feat_norm={fn:.4f}")
        if fn > best_norm:
            best_norm, best_key, best_feat = fn, name, feat

print(f"\n✅ Best ordering: '{best_key}'  (norm={best_norm:.4f})")

# Map best key → FUSION_MODE integer used in the model below
FUSION_MODE = {
    "A: norm(gated * gamma)":                    "A",
    "B: norm(gated) * gamma":                    "B",
    "C: norm(gated)":                            "C",
    "D: gated * gamma (no norm)":                "D",
    "E: norm(gate_out * gamma)":                 "E",
    "F: raw_comb -> gate -> norm*gamma":         "F",
}[best_key]
USE_L2_NORM = FUSION_MODE != "F"
print(f"   FUSION_MODE={FUSION_MODE}  USE_L2_NORM={USE_L2_NORM}")


# ════════════════════════════════════════════════════════════════
# DATASET
# ════════════════════════════════════════════════════════════════
class SeverityDataset(Dataset):
    def __init__(self, tsv_path, image_dir):
        self.df = pd.read_csv(tsv_path, sep='\t')
        self.image_dir = image_dir
        self.df['label_clean'] = self.df['damage_severity'].apply(self._clean_label)
        self.df = self.df.dropna(subset=['label_clean', 'image_path'])
        self.df = self.df[self.df['label_clean'].isin(target_classes)].reset_index(drop=True)
        self.label_to_idx = {label: idx for idx, label in enumerate(target_classes)}
        self.processor = CLIPProcessor.from_pretrained(
            "openai/clip-vit-base-patch32", use_fast=True)

    def _clean_label(self, label):
        try:
            if str(label).startswith('['):
                label_list = ast.literal_eval(label)
                val = str(label_list[0]) if label_list else "unknown"
            else:
                val = str(label)
            val = val.lower().strip()
            if "severe" in val:                                      return "severe_damage"
            if "mild"   in val:                                      return "mild_damage"
            if "none" in val or "no" in val or "little" in val:     return "little_or_no_damage"
            return val
        except:
            return "unknown"

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = os.path.join(self.image_dir, str(row['image_path']))
        label_key = row['label_clean']
        disaster  = str(row.get('disaster_types', 'disaster')).replace('_', ' ')
        text      = SEVERITY_TEXT_TEMPLATES[label_key].format(disaster=disaster)
        try:
            image  = Image.open(img_path).convert("RGB")
            inputs = self.processor(text=[text], images=image,
                                    return_tensors="pt", padding=True, truncation=True)
            return (inputs['pixel_values'].squeeze(0),
                    inputs['input_ids'].squeeze(0),
                    inputs['attention_mask'].squeeze(0),
                    torch.tensor(self.label_to_idx[label_key]))
        except:
            return None

def collate_fn(batch):
    batch = [x for x in batch if x is not None]
    if not batch: return None, None, None, None
    img, txt, mask, lbl = zip(*batch)
    return (torch.stack(img),
            torch.nn.utils.rnn.pad_sequence(txt,  batch_first=True),
            torch.nn.utils.rnn.pad_sequence(mask, batch_first=True),
            torch.stack(lbl))


# ════════════════════════════════════════════════════════════════
# MODEL — forward pass selected automatically by FUSION_MODE
# ════════════════════════════════════════════════════════════════
class CrisisCLIPModel(nn.Module):
    def __init__(self, num_classes=3, fusion_mode="A", use_l2_norm=True):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.use_l2_norm = use_l2_norm
        self.clip = CLIPModel.from_pretrained(
            "openai/clip-vit-base-patch32", use_safetensors=True)
        self.fusion = nn.ModuleDict({
            'gate': nn.Sequential(
                nn.Linear(1024, 512), nn.ReLU(),
                nn.Linear(512, 1024), nn.Sigmoid()
            ),
            'norm': nn.LayerNorm(1024),
        })
        self.fusion_gamma = nn.Parameter(torch.ones(1024))
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),       # 0
            nn.BatchNorm1d(512),        # 1
            nn.ReLU(),                  # 2
            nn.Dropout(0.3),            # 3
            nn.Linear(512, 128),        # 4
            nn.ReLU(),                  # 5
            nn.Linear(128, num_classes) # 6
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        out = self.clip(input_ids=input_ids,
                        attention_mask=attention_mask,
                        pixel_values=pixel_values)
        ie, te = out.image_embeds, out.text_embeds

        if self.use_l2_norm:
            ie = ie / ie.norm(dim=-1, keepdim=True)
            te = te / te.norm(dim=-1, keepdim=True)

        combined  = torch.cat([ie, te], dim=1)
        gate_out  = self.fusion['gate'](combined)
        gated     = gate_out * combined
        g         = self.fusion_gamma

        if   self.fusion_mode == "A": fused = self.fusion['norm'](gated * g)
        elif self.fusion_mode == "B": fused = self.fusion['norm'](gated) * g
        elif self.fusion_mode == "C": fused = self.fusion['norm'](gated)
        elif self.fusion_mode == "D": fused = gated * g
        elif self.fusion_mode == "E": fused = self.fusion['norm'](gate_out * g)
        elif self.fusion_mode == "F": fused = self.fusion['norm'](gated * g)

        return self.classifier(fused), fused


# ════════════════════════════════════════════════════════════════
# LOAD WEIGHTS
# ════════════════════════════════════════════════════════════════
lora_A, lora_B = {}, {}
for k, v in raw.items():
    canon = (k.replace("base_model.model.", "")
              .replace(".lora_A.default.weight", "")
              .replace(".lora_B.default.weight", ""))
    if   "lora_A" in k: lora_A[canon] = v
    elif "lora_B" in k: lora_B[canon] = v

clean = OrderedDict()
for k, v in raw.items():
    if "lora" in k: continue
    name = k.replace("base_model.model.", "").replace(".base_layer.", ".")
    base = name.replace(".weight", "")
    if base in lora_A and ".weight" in name:
        A = lora_A[base].to(device)
        B = lora_B[base].to(device)
        clean[name] = v.to(device) + (B @ A) * 1.0
    else:
        clean[name] = v
if "fusion.gamma" in clean:
    clean["fusion_gamma"] = clean.pop("fusion.gamma")

model = CrisisCLIPModel(
    num_classes=3, fusion_mode=FUSION_MODE, use_l2_norm=USE_L2_NORM
).to(device)
missing, unexpected = model.load_state_dict(clean, strict=False)
print(f"\n✅ Model loaded. Missing={len(missing)}, Unexpected={len(unexpected)}")
model.eval()

# ── Sanity check with the loaded model ──────────────────────────────
print("\nSanity check — 3 text probes:")
_blank_pil = Image.fromarray(np.ones((224,224,3), dtype=np.uint8)*128)
for _text in ["No damage, infrastructure intact.",
              "Minor structural damage.",
              "Catastrophic total destruction."]:
    _i = _proc(text=[_text], images=_blank_pil,
                return_tensors="pt", padding=True, truncation=True)
    _i = {k: v.to(device) for k, v in _i.items()}
    with torch.no_grad():
        _lg, _ft = model(_i['pixel_values'], _i['input_ids'], _i['attention_mask'])
        _pr = torch.softmax(_lg, dim=1).cpu().numpy()[0]
    print(f"  '{_text[:42]}' → {DISPLAY_NAMES[int(np.argmax(_pr))]:15s} "
          f"probs={_pr.round(3)}  feat_norm={_ft.norm():.3f}")


# ════════════════════════════════════════════════════════════════
# FULL INFERENCE
# ════════════════════════════════════════════════════════════════
print("\nLoading dataset...")
dataset    = SeverityDataset(DATASET_PATH, IMAGE_BASE_DIR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True, collate_fn=collate_fn)

all_preds, all_labels, all_probs, all_features = [], [], [], []

print(f"Testing {len(dataset)} samples...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        if batch[0] is None: continue
        pixel_values, input_ids, attention_mask, labels = [
            b.to(device, non_blocking=True) for b in batch
        ]
        logits, features = model(pixel_values, input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_features.extend(features.cpu().numpy())

all_labels   = np.array(all_labels)
all_preds    = np.array(all_preds)
all_probs    = np.array(all_probs)
all_features = np.array(all_features)

print("\nPrediction distribution:")
unique, counts = np.unique(all_preds, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  class {u} ({DISPLAY_NAMES[u]}): {c} ({100*c/len(all_preds):.1f}%)")


# ════════════════════════════════════════════════════════════════
# GRAPHS
# ════════════════════════════════════════════════════════════════
print("\n=== Task 3: Severity OOD Report ===")
print(classification_report(all_labels, all_preds,
                             target_names=DISPLAY_NAMES, zero_division=0))

# Graph 1: Confusion matrix
cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Reds",
            xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES)
plt.title("Normalized Confusion Matrix — Severity (OOD)")
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig("ood_cm_severity.png", dpi=300); plt.close()
print("Saved 1/3: ood_cm_severity.png")

# Graph 2: ROC curves
y_bin = label_binarize(all_labels, classes=[0, 1, 2])
plt.figure(figsize=(8, 6))
for i, (color, name) in enumerate(zip(['green','orange','red'], DISPLAY_NAMES)):
    if np.sum(y_bin[:, i]) > 0:
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'ROC {name} (AUC={auc(fpr,tpr):.2f})')
plt.plot([0,1],[0,1],'k--',lw=2)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Severity (OOD)')
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("ood_roc_severity.png", dpi=300); plt.close()
print("Saved 2/3: ood_roc_severity.png")

# Graph 3: t-SNE
print("Calculating t-SNE...")
idx = np.random.choice(len(all_features), min(1500, len(all_features)), replace=False)
tsne_r = TSNE(n_components=2, random_state=42,
              init='pca', learning_rate='auto').fit_transform(all_features[idx])
plt.figure(figsize=(9, 7))
sns.scatterplot(x=tsne_r[:,0], y=tsne_r[:,1],
                hue=[DISPLAY_NAMES[l] for l in all_labels[idx]],
                palette=sns.color_palette("Set1", 3), alpha=0.7, s=30)
plt.title("t-SNE Latent Space — Severity (OOD)")
plt.xlabel("TSNE-1"); plt.ylabel("TSNE-2")
plt.legend(title="Severity", bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.savefig("ood_tsne_severity.png", dpi=300); plt.close()
print("Saved 3/3: ood_tsne_severity.png")
print("Done!")

Tracing fusion forward pass to find correct ordering...
gamma  — min=0.000100  max=0.000100  mean=0.000100  norm=0.003200
norm.w — min=1.000000  max=1.000000  mean=1.000000
norm.b — min=0.000000  max=0.000000  mean=0.000000


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
from peft import LoraConfig, get_peft_model
import ast
from tqdm import tqdm

# --- CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 5e-5

# Make sure you have your train and validation TSV files ready!
TRAIN_DATASET_PATH = "MEDIC_train.tsv" 
VAL_DATASET_PATH = "MEDIC_dev.tsv"
IMAGE_BASE_DIR = r"F:\research\chatbot\MEDIC"
SAVE_MODEL_PATH = r"F:\research\chatbot\MEDIC\best_damage_model_retrained.pth"

target_classes = ["little or none", "mild", "severe"]

# --- DATASET HANDLING ---
SEVERITY_TEXT_TEMPLATES = {
    "little or none": "No visible damage from the {disaster}. Infrastructure appears intact.",
    "mild": "Minor damage visible from the {disaster}. Some structural issues present.",
    "severe": "Catastrophic and severe destruction from the {disaster}. Heavy damage everywhere."
}

class SeverityTrainDataset(Dataset):
    def __init__(self, tsv_path, image_dir):
        self.df = pd.read_csv(tsv_path, sep='\t')
        self.image_dir = image_dir

        self.df['label_clean'] = self.df['damage_severity'].apply(self._clean_label)
        self.df = self.df.dropna(subset=['label_clean', 'image_path'])
        self.df = self.df[self.df['label_clean'].isin(target_classes)].reset_index(drop=True)

        self.label_to_idx = {label: idx for idx, label in enumerate(target_classes)}
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

    def _clean_label(self, label):
        try:
            if str(label).startswith('['):
                label_list = ast.literal_eval(label)
                val = str(label_list[0]) if label_list else "unknown"
            else:
                val = str(label)
            val = val.lower().strip()
            if "severe" in val: return "severe"
            if "mild" in val: return "mild"
            if "none" in val or "no" in val or "little" in val: return "little or none"
            return val
        except:
            return "unknown"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['image_path']))

        label_key = row['label_clean']
        label_idx = self.label_to_idx[label_key]

        disaster = str(row.get('disaster_types', 'disaster')).replace('_', ' ')
        text_tmpl = SEVERITY_TEXT_TEMPLATES.get(label_key, "Damage from a {disaster}.")
        text = text_tmpl.format(disaster=disaster)

        try:
            image = Image.open(img_path).convert("RGB")
            inputs = self.processor(
                text=[text], images=image,
                return_tensors="pt", padding=True, truncation=True
            )
            return (
                inputs['pixel_values'].squeeze(0),
                inputs['input_ids'].squeeze(0),
                inputs['attention_mask'].squeeze(0),
                torch.tensor(label_idx)
            )
        except:
            return None

def collate_fn(batch):
    batch = [x for x in batch if x is not None]
    if not batch: return None, None, None, None
    img, txt, mask, lbl = zip(*batch)
    return (
        torch.stack(img),
        torch.nn.utils.rnn.pad_sequence(txt, batch_first=True),
        torch.nn.utils.rnn.pad_sequence(mask, batch_first=True),
        torch.stack(lbl)
    )

# --- ARCHITECTURE ---
class CrisisCLIPModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
        
        self.fusion = nn.ModuleDict({
            'text_proj': nn.Linear(512, 512),
            'image_proj': nn.Linear(512, 512),
            'gate': nn.Sequential(
                nn.Linear(1024, 512), nn.ReLU(),
                nn.Linear(512, 1024), nn.Sigmoid()
            )
        })
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128),  nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        outputs = self.clip(
            input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values
        )
        image_embeds = self.fusion['image_proj'](outputs.image_embeds)
        text_embeds  = self.fusion['text_proj'](outputs.text_embeds)
        combined     = torch.cat([image_embeds, text_embeds], dim=1)
        fused        = self.fusion['gate'](combined) * combined
        logits       = self.classifier(fused)
        return logits

# --- SETUP TRAINING ---
print("Initializing datasets...")
train_dataset = SeverityTrainDataset(TRAIN_DATASET_PATH, IMAGE_BASE_DIR)
val_dataset   = SeverityTrainDataset(VAL_DATASET_PATH, IMAGE_BASE_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collate_fn)

model = CrisisCLIPModel(num_classes=3)

# Apply LoRA specifically to the CLIP Backbone to save memory
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
)
# Wrap only the CLIP part in PEFT
model.clip = get_peft_model(model.clip, lora_config)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler() # AMP for faster/lighter training

best_val_f1 = 0.0

# --- TRAINING LOOP ---
print("\nStarting Training Phase...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in progress_bar:
        if batch[0] is None: continue
        pixel_values, input_ids, attention_mask, labels = [b.to(device, non_blocking=True) for b in batch]

        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            logits = model(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_train_loss = total_loss / len(train_loader)

    # --- VALIDATION LOOP ---
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  "):
            if batch[0] is None: continue
            pixel_values, input_ids, attention_mask, labels = [b.to(device, non_blocking=True) for b in batch]
            
            with torch.cuda.amp.autocast():
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)
                
            val_loss += loss.item()
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='weighted', zero_division=0)

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f}")

    # Checkpoint saving
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        print(f"New best model found (F1: {best_val_f1:.4f})! Saving weights...")
        
        # Save the full state dictionary so it loads cleanly next time
        torch.save(model.state_dict(), SAVE_MODEL_PATH)

print("\nTraining Complete! Best weights saved to:", SAVE_MODEL_PATH)

Initializing datasets...


C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:157: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # AMP for faster/lighter training



Starting Training Phase...


Epoch 1/5 [Train]:   0%|          | 0/1543 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/5 [Train]:  30%|██▉       | 456/1543 [08:11<25:01,  1.38s/it, loss=0.00672]d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/5 [Val]  :   0%|          | 0/193 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/5 [Val]  : 100%|██████████| 193/193 [04:52<00:00,  1.51s/it]



Epoch 1 Summary:
Train Loss: 0.0292 | Val Loss: 0.0001 | Val F1: 1.0000
New best model found (F1: 1.0000)! Saving weights...


Epoch 2/5 [Train]:   0%|          | 0/1543 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/5 [Train]:  13%|█▎        | 203/1543 [03:40<19:32,  1.14it/s, loss=0.000131]d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2/5 [Val]  :   0%|          | 0/193 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/5 [Val]  : 100%|██████████| 193/193 [03:27<00:00,  1.08s/it]



Epoch 2 Summary:
Train Loss: 0.0003 | Val Loss: 0.0000 | Val F1: 1.0000


Epoch 3/5 [Train]:   0%|          | 0/1543 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 3/5 [Train]:  37%|███▋      | 578/1543 [07:58<13:00,  1.24it/s, loss=3.44e-5] d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 3/5 [Val]  :   0%|          | 0/193 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 3/5 [Val]  : 100%|██████████| 193/193 [04:46<00:00,  1.48s/it]



Epoch 3 Summary:
Train Loss: 0.0001 | Val Loss: 0.0000 | Val F1: 1.0000


Epoch 4/5 [Train]:   0%|          | 0/1543 [00:00<?, ?it/s]C:\Users\nabil\AppData\Local\Temp\ipykernel_21112\4063051648.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 4/5 [Train]:  16%|█▌        | 242/1543 [03:26<21:12,  1.02it/s, loss=2.7e-5]  d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 4/5 [Train]:  35%|███▍      | 538/1543 [13:26<23:51,  1.42s/it, loss=3.35e-6] 